# Cross-agent results comparison

Loads every `TaskResult` record under `runs/` (the `RXX`-aliased run folders), dedupes to
the latest attempt per `(task_id, agent, modality)`, and demonstrates slicing by the
gui-failure-suite taxonomy (`split`, `app`, `benchmark_id`) and by `agent` / `status`.

The reserved `failure_category_id` column is `null` until joined with the external
failure-category repo — pass `failure_map=` to `load()` to populate it.

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable so `eval.helpers.load_results` resolves.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from eval.helpers.load_results import load

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [ ]:
# To attach failure categories from the external repo, pass failure_map:
#   df = load(failure_map="../path/to/failure_categories.json")
df = load()
print(f"{len(df)} records, {df['task_id'].nunique()} unique tasks")
df.head()

## Outcome rates per agent

In [ ]:
# Success rate is computed only over "scored" outcomes — runs where the agent
# actually attempted the task and produced a judgeable result. Non-outcomes
# (aborted / skipped / timeout) are excluded from the denominator, so a user
# interrupting a run or a harness timeout doesn't depress the rate.
SCORED_STATUSES = ["success", "failure", "error", "blocked", "human_failure"]

counts = (
    df.groupby(["agent", "status"]).size()
      .unstack(fill_value=0)
)
# Extra row pooling every agent's records into one combined set.
# Note: a task run by more than one agent is counted once per agent, not deduped.
counts.loc["combined"] = counts.sum(axis=0)

counts["total"] = counts.sum(axis=1)
scored_cols = [s for s in SCORED_STATUSES if s in counts.columns]
counts["scored"] = counts[scored_cols].sum(axis=1)
success = counts["success"] if "success" in counts.columns else 0
# 0/0 -> NaN (no scored attempts yet); guard against divide-by-zero noise.
counts["success_rate"] = (success / counts["scored"].replace(0, float("nan"))).round(3)
outcomes = counts
outcomes

## Task coverage and real-vs-custom app

How many tasks ran per `(agent, platform)`, and a split of real production apps/sites vs.
**custom** self-hosted environments (`localhost` web app / the `com.mazenbashammakh.mobile`
package). The Mobilerun real-vs-custom table holds the agent fixed so the outcome gap is
attributable to the target app rather than to the agent.

In [4]:
# Task coverage + real-vs-custom-app split. Reuses SCORED_STATUSES from the
# per-agent cell above. "Custom app" = self-hosted env (localhost / the custom
# com.mazenbashammakh.* package); everything else is a real production app/site.
print("=== records per (agent, platform) ===")
cov = df.groupby(["agent", "platform"]).size().unstack(fill_value=0)
cov["total"] = cov.sum(axis=1)
cov.loc["total"] = cov.sum(axis=0)
display(cov)

_app = df["app"].fillna("").astype(str)
df["app_kind"] = (
    _app.str.contains("localhost") | _app.str.contains("mazenbashammakh")
).map({True: "custom_app", False: "real_app"})
_plat = df["platform"].replace({"desktop_windows": "desktop"})

print("\n=== real vs custom app, by platform ===")
rc = pd.crosstab(_plat, df["app_kind"])
rc["total"] = rc.sum(axis=1)
rc.loc["total"] = rc.sum(axis=0)
display(rc)

# Same-agent real-vs-custom contrast: for Mobilerun the only thing that changes
# between these two rows is the target app, so this isolates the custom
# environment as a failure driver rather than confounding it with the agent.
print("\n=== Mobilerun: outcomes by app kind ===")
mr = df[df["agent"] == "mobilerun"]
mr_tbl = mr.groupby(["app_kind", "status"]).size().unstack(fill_value=0)
scored_cols = [s for s in SCORED_STATUSES if s in mr_tbl.columns]
mr_tbl["scored"] = mr_tbl[scored_cols].sum(axis=1)
mr_succ = mr_tbl["success"] if "success" in mr_tbl.columns else 0
mr_tbl["success_rate"] = (mr_succ / mr_tbl["scored"].replace(0, float("nan"))).round(3)
display(mr_tbl)

=== records per (agent, platform) ===


platform,desktop_windows,mobile,web,total
agent,,,,
agent_s,1,0,0,1
manual,0,0,21,21
mobilerun,0,69,0,69
seeact,0,0,32,32
total,1,69,53,123



=== real vs custom app, by platform ===


app_kind,custom_app,real_app,total
platform,,,
desktop,0,1,1
mobile,22,47,69
web,8,45,53
total,30,93,123



=== Mobilerun: outcomes by app kind ===


status,aborted,error,failure,human_failure,success,scored,success_rate
app_kind,,,,,,,
custom_app,1,0,1,11,9,21,0.429
real_app,4,1,2,4,36,43,0.837


## Outcome rates per modality (text_only / vision_only / multimodal)

Same scored-success computation as the per-agent table, but grouped by the `modality`
field (the perception modality resolved for each run). The `(modality, agent)` count
below shows how modality maps onto agents — if it's near 1:1, this view mostly re-buckets
the per-agent table rather than revealing modality as an independent factor.

In [5]:
# Reuses SCORED_STATUSES from the per-agent cell above. Rows with a missing
# modality (e.g. legacy runs / non-outcome records) are bucketed as "unknown"
# so they're visible rather than silently dropped.
mod = df.copy()
mod["modality"] = mod["modality"].fillna("unknown")

mod_counts = (
    mod.groupby(["modality", "status"]).size()
       .unstack(fill_value=0)
)
mod_counts["total"]  = mod_counts.sum(axis=1)
scored_cols = [s for s in SCORED_STATUSES if s in mod_counts.columns]
mod_counts["scored"] = mod_counts[scored_cols].sum(axis=1)
mod_success = mod_counts["success"] if "success" in mod_counts.columns else 0
# 0/0 -> NaN (no scored attempts yet); guard against divide-by-zero noise.
mod_counts["success_rate"] = (
    mod_success / mod_counts["scored"].replace(0, float("nan"))
).round(3)

print("=== outcomes by modality ===")
display(mod_counts)

# How modality maps onto agents — watch for near-1:1 (modality ~ agent alias).
print("\n=== records per (modality, agent) ===")
display(mod.groupby(["modality", "agent"]).size().unstack(fill_value=0))

=== outcomes by modality ===


status,aborted,blocked,error,failure,human_failure,success,total,scored,success_rate
modality,,,,,,,,,
multimodal,8,5,6,3,6,6,34,26,0.231
text_only,2,0,1,3,1,36,43,41,0.878
vision_only,3,0,0,0,14,9,26,23,0.391



=== records per (modality, agent) ===


agent,agent_s,manual,mobilerun,seeact
modality,,,,
multimodal,1,21,0,32
text_only,0,0,43,0
vision_only,0,0,26,0


## Slice by gui-failure-suite taxonomy (split / app / benchmark_id)

In [6]:
for axis in ["split", "app", "benchmark_id"]:
    if df[axis].notna().any():
        print(f"=== status by {axis} ===")
        print(df.groupby([axis, "status"]).size().unstack(fill_value=0))
        print()

=== status by split ===
status  aborted  blocked  error  failure  human_failure  success
split                                                           
test         13        5      7        6             21       50

=== status by app ===
status                                              aborted  blocked  error  failure  human_failure  success
app                                                                                                         
com.android.chrome                                        0        0      0        0              0        1
com.android.chrome, com.google.android.googlequ...        2        0      0        0              0       16
com.android.settings                                      0        0      0        0              0        1
com.android.settings, com.google.android.apps.maps        0        0      1        0              0        0
com.android.settings, com.twitter.android                 0        0      0        1              1     

## Latency and effort

In [7]:
df.groupby("agent")[["duration_s", "steps"]].describe()

duration_s                                                                                    steps                                                 
               count        mean         std        min         25%         50%         75%         max count       mean       std  min  25%   50%   75%   max
agent                                                                                                                                                         
agent_s          0.0         NaN         NaN        NaN         NaN         NaN         NaN         NaN   0.0        NaN       NaN  NaN  NaN   NaN   NaN   NaN
manual           0.0         NaN         NaN        NaN         NaN         NaN         NaN         NaN  21.0   2.238095  2.047065  0.0  1.0   1.0   3.0   8.0
mobilerun       49.0   62.912586   41.661858  22.160627   34.933686   50.508110   78.394494  216.478960  48.0   7.854167  6.321155  2.0  4.0   6.0   9.0  39.0
seeact          14.0  401.101316  233.598805   5.992686  260.475941  416.400666  560.703375  851.744339  14.0  10.642857  5.256602  0.0  8.0  11.5  15.0  17.0

## Step (effort) statistics

A deeper look at the `steps` column than the describe() above. Steps are only instrumented
for a subset of runs, so everything here is computed on the non-null subset. The cuts ask:
do failures cost more steps than successes, where does the total step budget go, how is
effort distributed across short vs. long episodes, what is the step *cost per success*, and
is there an apparent step ceiling that runs pile up against.

In [8]:
# === Detailed step (effort) statistics ===
import numpy as np

# Steps are only instrumented for some agents/runs. Operate on the non-null
# subset so missing instrumentation doesn't masquerade as 0-step episodes.
steps_df = df.dropna(subset=["steps"]).copy()
steps_df["steps"] = steps_df["steps"].astype(int)
print(f"steps recorded for {len(steps_df)}/{len(df)} records")
print("coverage by agent:")
display(steps_df.groupby("agent").size().rename("runs_with_steps").to_frame())

# 1) Per (agent, status): do failures burn more steps than successes?
print("\n=== steps by (agent, status) ===")
display(
    steps_df.groupby(["agent", "status"])["steps"]
            .agg(n="count", mean="mean", median="median", min="min", max="max")
            .round(2)
)

# 2) Efficiency contrast for the automated agents: successful episodes should be
#    shorter if the agent fails by thrashing / exhausting its step budget.
auto = steps_df[steps_df["agent"].isin(["seeact", "mobilerun"])].copy()
auto["outcome"] = np.where(auto["status"] == "success", "success", "non-success")
print("\n=== automated agents: steps, success vs non-success ===")
display(
    auto.groupby("outcome")["steps"]
        .agg(n="count", mean="mean", median="median", std="std")
        .round(2)
)

# 3) Step-budget allocation: of all effort actually expended, how much landed on
#    a success vs. was spent on a run that ultimately failed?
total = int(steps_df["steps"].sum())
budget = steps_df.groupby("status")["steps"].sum().sort_values(ascending=False).to_frame("total_steps")
budget["share_%"] = (budget["total_steps"] / total * 100).round(1)
print(f"\n=== step-budget allocation (total {total} steps spent) ===")
display(budget)

# 4) Effort distribution: how episodes spread across short vs. long runs.
bins, labels = [-1, 3, 8, 14, np.inf], ["quick (<=3)", "medium (4-8)", "long (9-14)", "very long (15+)"]
steps_df["effort"] = pd.cut(steps_df["steps"], bins=bins, labels=labels)
print("\n=== effort buckets x status ===")
display(pd.crosstab(steps_df["effort"], steps_df["status"], margins=True))

# 5) Cost-per-success (automated): total steps invested per task actually solved.
#    Lower is better; it folds wasted-on-failure effort into the success price.
costs = []
for ag, g in auto.groupby("agent"):
    n_succ = int((g["status"] == "success").sum())
    costs.append({"agent": ag, "total_steps": int(g["steps"].sum()),
                  "successes": n_succ,
                  "steps_per_success": round(g["steps"].sum() / n_succ, 2) if n_succ else np.nan})
print("\n=== steps invested per successful task (automated) ===")
display(pd.DataFrame(costs).set_index("agent"))

# 6) Apparent step ceiling: flag runs that hit the max observed for their agent.
print("\n=== runs at each agent's max observed step count (possible budget cap) ===")
mx = steps_df.groupby("agent")["steps"].transform("max")
display(steps_df[steps_df["steps"] == mx].groupby(["agent", "status"]).size().rename("n_at_max").to_frame())

steps recorded for 83/123 records
coverage by agent:


,runs_with_steps
agent,
manual,21
mobilerun,48
seeact,14



=== steps by (agent, status) ===


n   mean  median  min  max
agent     status                              
manual    success   1   7.00     7.0    7    7
mobilerun failure   3  15.67    15.0   10   22
          success  45   7.33     6.0    2   39
seeact    error     6   9.17    10.0    0   17
          failure   3  15.00    15.0   15   15
          success   5   9.80    10.0    8   13


=== automated agents: steps, success vs non-success ===


...

status,error,failure,success,All
effort,,,,
quick (<=3),2,0,5,7
medium (4-8),1,0,33,34
long (9-14),0,1,8,9
very long (15+),3,5,5,13
All,6,6,51,63



=== steps invested per successful task (automated) ===


,total_steps,successes,steps_per_success
agent,,,
mobilerun,377,45,8.38
seeact,149,5,29.80



=== runs at each agent's max observed step count (possible budget cap) ===


,,n_at_max
agent,status,
mobilerun,success,1
seeact,error,1


## Cross-agent comparison (same task, different agents)

Pivots `status` over the stable `task_id` join key. Rows where agents disagree are the
interesting cases for failure analysis.

In [9]:
pivot = df.pivot_table(
    index="task_id", columns="agent", values="status", aggfunc="first"
)
pivot.head(20)

agent,agent_s,manual,mobilerun,seeact
task_id,,,,
aitw-mobile-0001,NaN,NaN,success,NaN
aitw-mobile-0002,NaN,NaN,success,NaN
aitw-mobile-0003,NaN,NaN,success,NaN
aitw-mobile-0009,NaN,NaN,success,NaN
aitw-mobile-0014,NaN,NaN,success,NaN
aitw-mobile-0017,NaN,NaN,success,NaN
aitw-mobile-0023,NaN,NaN,success,NaN
aitw-mobile-0027,NaN,NaN,success,NaN
aitw-mobile-0034,NaN,NaN,aborted,NaN


## Failure category breakdown (once joined)

Empty until `load(failure_map=...)` is supplied from the external repo.

In [10]:
if df["failure_category_id"].notna().any():
    display(df.groupby(["failure_category_id", "agent"]).size().unstack(fill_value=0))
else:
    print("failure_category_id is all null — pass failure_map= to load() to populate it.")

failure_category_id is all null — pass failure_map= to load() to populate it.
